In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import h5py
import json
plt.style.use('./graph_preset.mplstyle')

In [2]:
read_path = Path("./0320104525_gp10/results.h5")
read_path = Path("./0405214835/results.h5")

In [3]:
with h5py.File(read_path, "r") as f: # read_paths[#] that you want to read
    print(f"--- Structure of {read_path} ---")

    def print_structure(name, obj):
        # データセットの場合は形状とデータ型も表示
        if isinstance(obj, h5py.Dataset):
            print(f"  {name} (Dataset) | Shape: {obj.shape}, Dtype: {obj.dtype}")
        # グループの場合はグループ名を表示
        elif isinstance(obj, h5py.Group):
            print(f"  {name} (Group)")

    f.visititems(print_structure)
    print("---------------------------------")

--- Structure of 0405214835\results.h5 ---
  input (Group)
  learning_curve (Group)
  output (Group)
  output/repeat_1 (Dataset) | Shape: (20, 18), Dtype: float32
  output/repeat_2 (Dataset) | Shape: (20, 18), Dtype: float32
  output/repeat_3 (Dataset) | Shape: (20, 18), Dtype: float32
  output/repeat_4 (Dataset) | Shape: (20, 18), Dtype: float32
  output/repeat_5 (Dataset) | Shape: (20, 18), Dtype: float32
---------------------------------


In [4]:
df_data = dict()

def store_dataset(name, obj):
    if isinstance(obj, h5py.Dataset):
        print(f"  Loading: {name} | Shape: {obj.shape}")
        df = pd.DataFrame(obj[:])
        df_data[name] = df

def store_dataset(name, obj):
    if isinstance(obj, h5py.Dataset):
        print(f"  Loading: {name} | Shape: {obj.shape}")

        # columns 属性があれば JSON から復元
        cols_attr = obj.attrs.get("columns", None)
        columns = None
        if cols_attr is not None:
            # 古い h5py だと bytes / np.bytes_ で返ることがある
            if isinstance(cols_attr, (bytes, np.bytes_)):
                cols_attr = cols_attr.decode("utf-8")
            columns = json.loads(cols_attr)

        # DataFrame 化（列名があれば使う）
        data = obj[:]  # dset[:, :] と同じ
        if columns is not None:
            df = pd.DataFrame(data, columns=columns)
        else:
            df = pd.DataFrame(data)

        df_data[name] = df


with h5py.File(read_path, "r") as f:
    print(f"--- Loading all datasets from {read_path} ---")
    f.visititems(store_dataset)
    print("---------------------------------------------")

print("\n--- Dictionary Keys ---")
print(list(df_data.keys()))
print("-----------------------")

--- Loading all datasets from 0405214835\results.h5 ---
  Loading: output/repeat_1 | Shape: (20, 18)
  Loading: output/repeat_2 | Shape: (20, 18)
  Loading: output/repeat_3 | Shape: (20, 18)
  Loading: output/repeat_4 | Shape: (20, 18)
  Loading: output/repeat_5 | Shape: (20, 18)
---------------------------------------------

--- Dictionary Keys ---
['output/repeat_1', 'output/repeat_2', 'output/repeat_3', 'output/repeat_4', 'output/repeat_5']
-----------------------


In [5]:
pd.set_option('display.max_rows', None)

In [6]:
df_data["output/repeat_1"]

,h1,h2,h3,h4,h5,s1,s2,s3,s4,s5,a,b,k,S11,Metric,gamma,routine_idx,best
0,5.000000,1.503409,2.285533,2.441034,4.919906,0.000000,1.500000,1.500000,1.500000,1.500000,2.000000,3.627538,6.000000,-21.980181,NaN,NaN,0.0,-21.980181
1,3.427682,9.181537,8.223880,5.828424,5.814837,1.350239,0.714623,0.943388,0.981410,1.477707,2.849617,2.247849,3.242599,-3.303322,NaN,NaN,0.0,-21.980181
2,5.977857,3.733006,7.972189,8.670664,5.676379,1.847790,1.535293,1.410771,0.224597,1.007669,2.342554,6.080307,2.596411,-3.369010,NaN,NaN,0.0,-21.980181
3,2.585186,5.240226,6.641131,6.905287,1.907362,1.291876,1.640393,0.215454,0.043556,0.331742,4.704216,4.484449,1.099633,-3.295846,NaN,NaN,0.0,-21.980181
4,7.133537,1.363935,1.471619,6.232123,9.461607,1.254403,0.596002,0.825381,0.421417,1.218508,5.627521,3.589781,1.794111,-3.207783,NaN,NaN,0.0,-21.980181
5,1.298062,9.557651,3.318474,9.302751,8.244004,1.475613,0.181598,0.554156,1.409446,1.791479,7.418167,5.661223,3.512983,-9.224373,NaN,NaN,0.0,-21.980181
6,1.809485,7.869865,3.723738,2.108133,4.830314,1.583397,1.836394,0.399657,1.251020,1.968425,6.489630,7.499846,4.278742,-4.374350,NaN,NaN,0.0,-21.980181
7,4.805021,4.138103,7.413072,1.436764,3.804997,1.642736,1.003816,1.291758,1.490523,1.420746,7.311635,5.281012,1.708561,-4.610034,NaN,NaN,0.0,-21.980181
8,9.636924,8.492733,4.517991,7.262146,4.665742,1.558020,0.381500,0.052929,0.196517,1.119912,3.896305,3.220762,2.083234,-2.775671,NaN,NaN,0.0,-21.980181
9,4.484871,3.160350,3.920970,3.848002,1.970582,1.952922,1.955267,0.986765,1.126972,0.018226,6.048900,7.936216,4.711437,-2.621559,NaN,NaN,0.0,-21.980181


In [7]:
combined = pd.concat(df_data, names=["source"])
min_pos = combined["S11"].idxmin()

print(f"Min S11: {combined.loc[min_pos, 'S11']}")
print(f"Location: {min_pos}")
print("That row:")
print(combined.loc[min_pos])

Min S11: -21.980180740356445
Location: ('output/repeat_1', 0)
That row:
h1              5.000000
h2              1.503409
h3              2.285533
h4              2.441034
h5              4.919906
s1              0.000000
s2              1.500000
s3              1.500000
s4              1.500000
s5              1.500000
a               2.000000
b               3.627538
k               6.000000
S11           -21.980181
Metric               NaN
gamma                NaN
routine_idx     0.000000
best          -21.980181
Name: (output/repeat_1, 0), dtype: float32
